# 📊 Evaluation Results — Email Writing Assistant

Visualizes evaluation metrics comparing the **base** Llama 3.1 8B Instruct model vs the **fine-tuned** model.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set style
plt.style.use('dark_background')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 12

In [ ]:
# Load evaluation results
RESULTS_FILE = "evaluation/results.json"  # Adjust path as needed

with open(RESULTS_FILE, 'r') as f:
    results = json.load(f)

ft_metrics = results['fine_tuned']['metrics']
base_metrics = results.get('base', {}).get('metrics', {})

print("Fine-tuned metrics loaded:", len(ft_metrics), "keys")
if base_metrics:
    print("Base model metrics loaded:", len(base_metrics), "keys")

## 1. N-gram Overlap Metrics (ROUGE & BLEU)

In [ ]:
# Bar chart: ROUGE & BLEU comparison
overlap_keys = ['rouge1', 'rouge2', 'rougeL', 'bleu']
overlap_labels = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BLEU']

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(overlap_keys))
width = 0.35

ft_vals = [ft_metrics.get(k, 0) for k in overlap_keys]
bars1 = ax.bar(x + width/2, ft_vals, width, label='Fine-Tuned', color='#818cf8', edgecolor='white', linewidth=0.5)

if base_metrics:
    base_vals = [base_metrics.get(k, 0) for k in overlap_keys]
    bars2 = ax.bar(x - width/2, base_vals, width, label='Base Model', color='#475569', edgecolor='white', linewidth=0.5)

ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('N-gram Overlap Metrics', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(overlap_labels)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('evaluation/rouge_bleu_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Semantic Similarity (BERTScore)

In [ ]:
# BERTScore components
bert_keys = ['bertscore_precision', 'bertscore_recall', 'bertscore_f1']
bert_labels = ['Precision', 'Recall', 'F1']

fig, ax = plt.subplots(figsize=(8, 6))

ft_bert = [ft_metrics.get(k, 0) for k in bert_keys]
colors = ['#a78bfa', '#818cf8', '#6366f1']
bars = ax.bar(bert_labels, ft_bert, color=colors, edgecolor='white', linewidth=0.5, width=0.5)

ax.set_ylabel('Score')
ax.set_title('BERTScore Components (Fine-Tuned)', fontsize=16, fontweight='bold')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.4f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.savefig('evaluation/bertscore.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Email Format Compliance

In [ ]:
# Format compliance breakdown
format_keys = ['has_greeting_rate', 'has_body_rate', 'has_closing_rate', 'format_compliance']
format_labels = ['Has Greeting', 'Has Body', 'Has Closing', 'Overall Compliance']

ft_format = [ft_metrics.get(k, 0) for k in format_keys]

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#34d399', '#60a5fa', '#f472b6', '#fbbf24']
bars = ax.barh(format_labels, ft_format, color=colors, edgecolor='white', linewidth=0.5, height=0.5)

ax.set_xlabel('Rate')
ax.set_title('Email Format Compliance (Fine-Tuned)', fontsize=16, fontweight='bold')
ax.set_xlim(0, 1.1)
ax.grid(axis='x', alpha=0.3)

for bar in bars:
    width = bar.get_width()
    ax.annotate(f'{width:.1%}', xy=(width, bar.get_y() + bar.get_height()/2),
                xytext=(5, 0), textcoords='offset points', ha='left', va='center', fontsize=11)

plt.tight_layout()
plt.savefig('evaluation/format_compliance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Lexical Diversity (Distinct-n)

In [ ]:
# Distinct-n scores
distinct_keys = ['distinct_1', 'distinct_2', 'distinct_3']
distinct_labels = ['Distinct-1', 'Distinct-2', 'Distinct-3']

fig, ax = plt.subplots(figsize=(8, 5))

ft_distinct = [ft_metrics.get(k, 0) for k in distinct_keys]
ax.plot(distinct_labels, ft_distinct, 'o-', color='#818cf8', linewidth=2, markersize=10, label='Fine-Tuned')

if base_metrics:
    base_distinct = [base_metrics.get(k, 0) for k in distinct_keys]
    ax.plot(distinct_labels, base_distinct, 's--', color='#475569', linewidth=2, markersize=10, label='Base')

ax.set_ylabel('Diversity Score')
ax.set_title('Lexical Diversity (Distinct-n)', fontsize=16, fontweight='bold')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation/diversity.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Length Distribution Comparison

In [ ]:
# Word count comparison (generated vs reference)
fig, ax = plt.subplots(figsize=(8, 5))

categories = ['Word Count', 'Sentence Count', 'Char Count']
pred_means = [
    ft_metrics.get('pred_word_count_mean', 0),
    ft_metrics.get('pred_sentence_count_mean', 0),
    ft_metrics.get('pred_char_count_mean', 0) / 10,  # Scale chars for visibility
]
ref_means = [
    ft_metrics.get('ref_word_count_mean', 0),
    ft_metrics.get('ref_sentence_count_mean', 0),
    ft_metrics.get('ref_char_count_mean', 0) / 10,
]

x = np.arange(len(categories))
width = 0.3

ax.bar(x - width/2, pred_means, width, label='Generated', color='#818cf8')
ax.bar(x + width/2, ref_means, width, label='Reference', color='#34d399')

ax.set_xticks(x)
ax.set_xticklabels(['Words', 'Sentences', 'Chars (÷10)'])
ax.set_title('Length Distribution: Generated vs Reference', fontsize=16, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation/length_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Sample Generated Emails

In [ ]:
# Display sample generated vs reference emails
per_sample = results['fine_tuned']['per_sample']

for i, sample in enumerate(per_sample[:5]):
    print(f"\n{'='*60}")
    print(f"Sample {i+1} | Task: {sample.get('task_type', 'N/A')}")
    print(f"{'='*60}")
    print(f"\n📝 Prompt: {sample['prompt']}")
    print(f"\n🤖 Generated:\n{sample['prediction']}")
    print(f"\n✅ Reference:\n{sample['reference']}")
    print(f"\n📊 ROUGE-1: {sample.get('rouge1', 'N/A'):.4f} | ROUGE-L: {sample.get('rougeL', 'N/A'):.4f}")

## 7. Summary Table

In [ ]:
# Summary comparison table
summary_keys = [
    'rouge1', 'rouge2', 'rougeL', 'bleu',
    'bertscore_f1', 'distinct_1', 'distinct_2',
    'format_compliance',
]

data = []
for k in summary_keys:
    row = {'Metric': k}
    row['Fine-Tuned'] = f"{ft_metrics.get(k, 0):.4f}"
    if base_metrics:
        row['Base'] = f"{base_metrics.get(k, 0):.4f}"
        delta = ft_metrics.get(k, 0) - base_metrics.get(k, 0)
        row['Δ'] = f"{'+' if delta >= 0 else ''}{delta:.4f}"
    data.append(row)

df = pd.DataFrame(data)
print("\n" + df.to_string(index=False))

## 8. Training Loss Curve

In [ ]:
# Load TensorBoard logs manually (alternative to TensorBoard UI)
try:
    from tensorboard.backend.event_processing import event_accumulator
    import glob
    
    log_dirs = glob.glob('logs/events*') or glob.glob('../logs/events*')
    if log_dirs:
        ea = event_accumulator.EventAccumulator(log_dirs[0])
        ea.Reload()
        
        if 'train/loss' in ea.Tags()['scalars']:
            loss_events = ea.Scalars('train/loss')
            steps = [e.step for e in loss_events]
            losses = [e.value for e in loss_events]
            
            fig, ax = plt.subplots(figsize=(10, 5))
            ax.plot(steps, losses, color='#818cf8', linewidth=1.5)
            ax.set_xlabel('Step')
            ax.set_ylabel('Loss')
            ax.set_title('Training Loss', fontsize=16, fontweight='bold')
            ax.grid(alpha=0.3)
            plt.tight_layout()
            plt.savefig('evaluation/training_loss.png', dpi=150, bbox_inches='tight')
            plt.show()
        else:
            print("No training loss found in TensorBoard logs.")
    else:
        print("No TensorBoard log files found. Run training first.")
except ImportError:
    print("Install tensorboard to visualize training curves: pip install tensorboard")